In [1]:
!pip install transformers torch xgboost hf_xet tqdm

In [34]:
import pandas as pd
import numpy as np
import re
import string
from datetime import datetime
import warnings
import os
from email import message_from_string
warnings.filterwarnings('ignore')
import seaborn as sns
import matplotlib.pyplot as plt
from wordcloud import WordCloud
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from scipy.sparse import hstack
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
import torch
from transformers import BertTokenizer, BertModel
from tqdm import tqdm
from transformers import AutoTokenizer
from torch.utils.data import TensorDataset, Dataset, DataLoader
import torch.nn as nn
import torch.optim as optim
from transformers import AutoTokenizer, AutoModel

In [3]:
combined_df_csv = r"D:\Northeastern\Fall2025\DS5500\Spam_Email_Detection\notebooks\df_combined.csv"

In [4]:
df = pd.read_csv(combined_df_csv)

In [5]:
df.shape

(130234, 6)

In [6]:
df = df.dropna(how='any')
df.reset_index(drop=True, inplace=True)

In [7]:
df.shape

(124352, 6)

In [8]:
df.head()

,sender,receiver,subject,body,urls,category
0,cPanel <service@cpanel.com>,jose@monkey.org,Verify Your Account,Business with \t\t\t\t\t\t\t\tcPanel & WHM \t...,1.0,phishing
1,Ann Garcia <AnGarcia@mcoe.org>,"""info@maaaaa.org"" <info@maaaaa.org>",IT-Service Help Desk,Password will expire in 3 days. Click Here To ...,0.0,phishing
2,"""USAA"" <usaaacctupdate@sccu4u.com>",Recipients <usaaacctupdate@sccu4u.com>,Final USAA Reminder - Update Your Account Now,"To ensure delivery to your inbox, please add U...",1.0,phishing
3,=?utf-8?Q?P=2eSupport?= <infos@2015p.com>,jose@monkey.org,=?utf-8?Q?Dear=20Client=20=3a=20Update=20Your=...,"PayPal Secure Dear Client, ...",1.0,phishing
4,Update,Recipients,Update,08/10/2011 06:14:36 am +0800 - en_US.UTF-8 - O...,1.0,phishing


In [9]:
df = df[['sender', 'subject', 'body', 'category']]

In [10]:
df['sender_domain'] = df['sender'].astype(str).str.extract(r'@([\w\.-]+)')
df['sender_domain'] = df['sender_domain'].fillna(df['sender']).str.lower()

In [11]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'http\S+', 'url', text)         
    text = re.sub(r'\d+', 'number', text)          
    text = text.translate(str.maketrans('', '', string.punctuation))
    text = re.sub(r'\s+', ' ', text).strip()     
    return text

df['subject'] = df['subject'].apply(clean_text)
df['body'] = df['body'].apply(clean_text)

In [12]:
df.head(3)

,sender,subject,body,category,sender_domain
0,cPanel <service@cpanel.com>,verify your account,business with cpanel whm dear client our techn...,phishing,cpanel.com
1,Ann Garcia <AnGarcia@mcoe.org>,itservice help desk,password will expire in number days click here...,phishing,mcoe.org
2,"""USAA"" <usaaacctupdate@sccu4u.com>",final usaa reminder update your account now,to ensure delivery to your inbox please add us...,phishing,sccu4u.com


In [13]:
X_train_text, X_test_text, y_train, y_test = train_test_split(
    df[['sender_domain', 'subject', 'body']], df['category'],
    test_size=0.2, random_state=42, stratify=df['category']
)

In [14]:
vectorizer_domain = TfidfVectorizer(max_features=1000)
vectorizer_subject = TfidfVectorizer(max_features=2000)
vectorizer_body = TfidfVectorizer(max_features=8000)

In [15]:
X_domain_train = vectorizer_domain.fit_transform(X_train_text['sender_domain'])
X_subject_train = vectorizer_subject.fit_transform(X_train_text['subject'])
X_body_train = vectorizer_body.fit_transform(X_train_text['body'])

X_domain_test = vectorizer_domain.transform(X_test_text['sender_domain'])
X_subject_test = vectorizer_subject.transform(X_test_text['subject'])
X_body_test = vectorizer_body.transform(X_test_text['body'])

In [16]:
# X_train, X_test, y_train, y_test = train_test_split(X_all, y, test_size=0.2, random_state=42)

In [17]:
X_train = hstack([X_domain_train, X_subject_train, X_body_train])
X_test  = hstack([X_domain_test, X_subject_test, X_body_test])

In [18]:
log_model = LogisticRegression(max_iter=2000)
log_model.fit(X_train, y_train)

y_pred_log = log_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_log))
print(classification_report(y_test, y_pred_log))

Accuracy: 0.9785292107273531
              precision    recall  f1-score   support

       legit       0.99      0.99      0.99     13451
    phishing       0.80      0.82      0.81       634
        spam       0.97      0.98      0.98     10786

    accuracy                           0.98     24871
   macro avg       0.92      0.93      0.93     24871
weighted avg       0.98      0.98      0.98     24871



In [19]:
nb_model = MultinomialNB()
nb_model.fit(X_train, y_train)

y_pred_nb = nb_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_nb))
print(classification_report(y_test, y_pred_nb))

Accuracy: 0.9636926540951308
              precision    recall  f1-score   support

       legit       0.98      0.98      0.98     13451
    phishing       0.67      0.90      0.77       634
        spam       0.97      0.95      0.96     10786

    accuracy                           0.96     24871
   macro avg       0.87      0.94      0.90     24871
weighted avg       0.97      0.96      0.96     24871



In [20]:
svm_model = LinearSVC()
svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_svm))
print(classification_report(y_test, y_pred_svm))

Accuracy: 0.9806199991958506
              precision    recall  f1-score   support

       legit       0.99      0.99      0.99     13451
    phishing       0.82      0.87      0.84       634
        spam       0.98      0.98      0.98     10786

    accuracy                           0.98     24871
   macro avg       0.93      0.94      0.94     24871
weighted avg       0.98      0.98      0.98     24871



In [21]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))

Accuracy: 0.9731414096739174
              precision    recall  f1-score   support

       legit       0.98      0.99      0.98     13451
    phishing       0.83      0.72      0.77       634
        spam       0.97      0.97      0.97     10786

    accuracy                           0.97     24871
   macro avg       0.93      0.89      0.91     24871
weighted avg       0.97      0.97      0.97     24871



In [22]:
encoder = LabelEncoder()
y_train_enc = encoder.fit_transform(y_train)
y_test_enc = encoder.transform(y_test)

In [23]:
xgb_model = XGBClassifier(use_label_encoder=False, eval_metric='mlogloss', random_state=42)
xgb_model.fit(X_train, y_train_enc)

y_pred_xgb = xgb_model.predict(X_test)
y_pred_labels = encoder.inverse_transform(y_pred_xgb)

print("Accuracy:", accuracy_score(y_test_enc, y_pred_xgb))
print(classification_report(y_test_enc, y_pred_xgb, target_names=encoder.classes_))

Accuracy: 0.9733022395561095
              precision    recall  f1-score   support

       legit       0.99      0.98      0.98     13451
    phishing       0.81      0.82      0.82       634
        spam       0.97      0.98      0.97     10786

    accuracy                           0.97     24871
   macro avg       0.92      0.93      0.92     24871
weighted avg       0.97      0.97      0.97     24871



In [24]:
# tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
# bert_model = BertModel.from_pretrained('bert-base-uncased')

In [25]:
df['combined_text'] = (
    df['sender_domain'].astype(str) + " [SEP] " +
    df['subject'].astype(str) + " [SEP] " +
    df['body'].astype(str)
)

In [26]:
tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [27]:
label_enc = LabelEncoder()
df['label'] = label_enc.fit_transform(df['category'])

In [28]:
train_texts, test_texts, y_train, y_test = train_test_split(
    df['combined_text'].tolist(),
    df['label'].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df['label']
)

In [29]:
max_len = 256
train_enc = tokenizer(
    train_texts,
    truncation=True,
    padding=True,
    max_length=max_len,
    return_tensors='pt'
)
test_enc = tokenizer(
    test_texts,
    truncation=True,
    padding=True,
    max_length=max_len,
    return_tensors='pt'
)

In [30]:
y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [31]:
train_dataset = TensorDataset(train_enc['input_ids'], train_enc['attention_mask'], y_train_tensor)
test_dataset  = TensorDataset(test_enc['input_ids'], test_enc['attention_mask'], y_test_tensor)

In [32]:
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=16)

In [36]:
class BertCNNClassifier(nn.Module):
    def __init__(self, model_name, num_classes, freeze_bert=False):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        hidden_size = self.bert.config.hidden_size

        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=hidden_size, out_channels=128, kernel_size=k)
            for k in [2, 3, 4]
        ])
        self.dropout = nn.Dropout(0.3)
        self.fc = nn.Linear(128 * len(self.convs), num_classes)

        if freeze_bert:
            for param in self.bert.parameters():
                param.requires_grad = False

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        last_hidden_state = outputs.last_hidden_state       # [batch, seq_len, hidden_size]

        x = last_hidden_state.permute(0, 2, 1)              # [batch, hidden_size, seq_len]
        x = [torch.relu(conv(x)) for conv in self.convs]    # list of [batch, 128, L']
        x = [torch.max(pool, dim=2)[0] for pool in x]       # global max-pooling
        x = torch.cat(x, dim=1)                             # [batch, 128*3]
        x = self.dropout(x)
        out = self.fc(x)
        return out

In [38]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = len(label_enc.classes_)
model = BertCNNClassifier("distilbert-base-uncased", num_classes, freeze_bert=False).to(device)

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [39]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(model.parameters(), lr=2e-5)
epochs = 5

In [ ]:
for epoch in range(epochs):
    model.train()
    total_loss = 0
    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}"):
        input_ids, attention_mask, labels = [x.to(device) for x in batch]

        optimizer.zero_grad()
        outputs = model(input_ids, attention_mask)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    print(f"Epoch {epoch+1} | Training Loss: {total_loss/len(train_loader):.4f}")

Epoch 1/5:   1%|▊                                                                  | 74/6218 [03:27<4:45:32,  2.79s/it]

In [ ]:
model.eval()
y_true, y_pred = [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        outputs = model(input_ids, attention_mask)
        preds = torch.argmax(outputs, dim=1)
        y_pred.extend(preds.cpu().numpy())
        y_true.extend(labels.cpu().numpy())

print("Accuracy:", accuracy_score(y_true, y_pred))
print(classification_report(y_true, y_pred, target_names=label_enc.classes_))